# Build Yearly Valhalla Graphs

Create the WSL working layout, copy each historical PBF into WSL, build one Valhalla graph per year with Docker, validate it with a test route, and write restartable status files.

This notebook assumes WSL is initially empty except for Docker being available inside WSL.

## What This Notebook Does

- Creates `/home/<wsl-user>/gruendungsanalyse` in WSL.
- Creates `data/routing/valhalla_graphs/{year}` folders.
- Copies `TOOLS/osm-data/austria-YY0101.osm.pbf` into the matching WSL graph folder.
- Starts `ghcr.io/valhalla/valhalla-scripted:latest` with that folder mounted as `/custom_files`.
- Waits until Valhalla responds on `http://localhost:8002`.
- Runs a Graz test route and writes `build_manifest.json`.
- Stops the container and leaves the graph files in WSL for later reuse.

The first graph build can take a long time. Run one year first, usually 2025, before launching all years.

In [7]:
from __future__ import annotations

from datetime import datetime
from pathlib import Path
import hashlib
import json
import os
import subprocess
import time

import pandas as pd
import requests


def find_project_dir(start: Path) -> Path:
    for path in [start.resolve(), *start.resolve().parents]:
        if (path / "README.md").exists() and (path / "ANAL").exists() and (path / "TOOLS").exists():
            return path
    raise FileNotFoundError("Could not find project root")


PROJECT_DIR = find_project_dir(Path.cwd())
OSM_DIR = PROJECT_DIR / "TOOLS" / "osm-data"
STATUS_DIR = PROJECT_DIR / "ANAL" / "data" / "routing" / "status"
STATUS_DIR.mkdir(parents=True, exist_ok=True)

YEARS = list(range(2015, 2026))
PBF_BY_YEAR = {year: OSM_DIR / f"austria-{str(year)[2:]}0101.osm.pbf" for year in YEARS}
GRAPH_STATUS_PATH = STATUS_DIR / "graph_build_status.csv"

VALHALLA_IMAGE = "ghcr.io/valhalla/valhalla-scripted:latest"
VALHALLA_URL = "http://localhost:8002"
VALHALLA_CPUS = 16
WSL_EXE = r"C:\Windows\System32\wsl.exe"

# Set this to the distro name shown by `wsl -l -v` in PowerShell.
# Common values are "Ubuntu" or "Ubuntu-24.04". Use None only if your default WSL distro is already correct.
WSL_DISTRO = "Ubuntu"

WSL_PROJECT_ROOT = "$HOME/gruendungsanalyse"
WSL_GRAPH_ROOT = f"{WSL_PROJECT_ROOT}/data/routing/valhalla_graphs"

# Safety switch. Set to True only when you really want to start Docker builds.
RUN_BUILD = True

# Start with one smoke-test year. Change to YEARS after the 2025 build validates.
BUILD_YEARS = YEARS

# Existing valid manifests are skipped unless this is True.
OVERWRITE = False

# Austria graphs can take a while to build. Increase if your machine is slow.
MAX_WAIT_MINUTES = 180

# Graz city test route, lon/lat WGS84.
TEST_ROUTE = [
    {"lon": 15.4395, "lat": 47.0707},
    {"lon": 15.4630, "lat": 47.0580},
]

In [8]:
def run_local(command: list[str], check: bool = True, capture_output: bool = True) -> subprocess.CompletedProcess:
    return subprocess.run(command, check=check, text=True, capture_output=capture_output)


def wsl_base_command() -> list[str]:
    if WSL_DISTRO:
        return [WSL_EXE, "-d", WSL_DISTRO, "--"]
    return [WSL_EXE, "--"]


def run_wsl(command: str, check: bool = True, capture_output: bool = True) -> subprocess.CompletedProcess:
    return run_local([*wsl_base_command(), "bash", "-lc", command], check=check, capture_output=capture_output)


def require_wsl_distribution() -> None:
    distro_list = run_local([WSL_EXE, "-l", "-q"], check=False)
    combined_output = f"{distro_list.stdout}\n{distro_list.stderr}".lower()
    no_distribution = "no installed distributions" in combined_output or "has no installed distributions" in combined_output
    if distro_list.returncode != 0 and no_distribution:
        raise RuntimeError(
            "WSL is installed, but no Linux distribution is installed yet. "
            "Install Ubuntu first with `wsl --install -d Ubuntu`, restart if Windows asks, "
            "open Ubuntu once to create your Linux user, then rerun this notebook."
        )
    available_distros = [line.strip().replace("\x00", "") for line in distro_list.stdout.splitlines() if line.strip().replace("\x00", "")]
    if WSL_DISTRO and available_distros and WSL_DISTRO not in available_distros:
        raise RuntimeError(
            f"Configured WSL_DISTRO={WSL_DISTRO!r}, but PowerShell reports these WSL distros: {available_distros}. "
            "Set WSL_DISTRO in the config cell to the exact name from `wsl -l -v`."
        )

    test = run_wsl("printf ok", check=False)
    if test.returncode != 0:
        raise RuntimeError(
            "Could not start the configured WSL distribution. Run `wsl -l -v` in PowerShell "
            "and set WSL_DISTRO in this notebook to the exact distro name.\n\n"
            f"stdout:\n{test.stdout}\n\nstderr:\n{test.stderr}"
        )


def quote_bash(value: str) -> str:
    return "'" + value.replace("'", "'\\''") + "'"


def quote_wsl_path(value: str) -> str:
    if value.startswith("$HOME/"):
        return "$HOME/" + quote_bash(value.removeprefix("$HOME/"))
    return quote_bash(value)


def windows_to_wsl_path(path: Path) -> str:
    result = run_local([*wsl_base_command(), "wslpath", "-a", str(path)], check=False)
    if result.returncode == 0 and result.stdout.strip():
        return result.stdout.strip()

    resolved = path.resolve()
    drive = resolved.drive.rstrip(":").lower()
    if drive:
        relative = resolved.relative_to(resolved.anchor).as_posix()
        return f"/mnt/{drive}/{relative}"
    raise RuntimeError(f"Could not convert Windows path to WSL path: {path}\n{result.stderr}")


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def read_status() -> pd.DataFrame:
    if GRAPH_STATUS_PATH.exists():
        return pd.read_csv(GRAPH_STATUS_PATH)
    return pd.DataFrame(columns=["year", "status", "started_at", "finished_at", "duration_minutes", "graph_path_wsl", "manifest_path_wsl", "error_message"])


def write_status_row(row: dict) -> None:
    status = read_status()
    status = status[status["year"] != row["year"]]
    status = pd.concat([status, pd.DataFrame([row])], ignore_index=True)
    status = status.sort_values("year")
    status.to_csv(GRAPH_STATUS_PATH, index=False)


def wsl_graph_dir(year: int) -> str:
    return f"{WSL_GRAPH_ROOT}/{year}"


def wsl_manifest_path(year: int) -> str:
    return f"{wsl_graph_dir(year)}/build_manifest.json"


def container_name(year: int) -> str:
    return f"co2-valhalla-{year}"

## Preflight Checks

Run this cell first. It confirms that WSL and Docker are reachable and creates the base WSL directory layout.

In [9]:
missing_pbf = [path for path in PBF_BY_YEAR.values() if not path.exists()]
if missing_pbf:
    raise FileNotFoundError(missing_pbf[0])

require_wsl_distribution()

whoami = run_wsl("whoami").stdout.strip()
docker_version = run_wsl("docker --version").stdout.strip()
run_wsl(f"mkdir -p {quote_wsl_path(WSL_GRAPH_ROOT)} {quote_wsl_path(WSL_PROJECT_ROOT + '/logs/graph_builder')} {quote_wsl_path(WSL_PROJECT_ROOT + '/status')}")

print(f"WSL user: {whoami}")
print(docker_version)
print(f"WSL project root: {WSL_PROJECT_ROOT}")
print(f"Graph root: {WSL_GRAPH_ROOT}")

WSL user: leo
Docker version 29.5.3, build d1c06ef
WSL project root: $HOME/gruendungsanalyse
Graph root: $HOME/gruendungsanalyse/data/routing/valhalla_graphs


In [10]:
def graph_manifest_exists(year: int) -> bool:
    test_command = f"test -f {quote_wsl_path(wsl_manifest_path(year))}"
    return run_wsl(test_command, check=False).returncode == 0


def copy_pbf_to_wsl(year: int) -> str:
    source_windows = PBF_BY_YEAR[year]
    source_wsl = windows_to_wsl_path(source_windows)
    graph_dir = wsl_graph_dir(year)
    target_wsl = f"{graph_dir}/{source_windows.name}"
    command = " && ".join([
        f"mkdir -p {quote_wsl_path(graph_dir)}",
        f"test -f {quote_wsl_path(target_wsl)} || cp {quote_bash(source_wsl)} {quote_wsl_path(target_wsl)}",
    ])
    run_wsl(command)
    return target_wsl


def start_valhalla_container(year: int) -> None:
    name = container_name(year)
    graph_dir = wsl_graph_dir(year)
    command = " && ".join([
        f"docker rm -f {quote_bash(name)} >/dev/null 2>&1 || true",
        "docker run -d "
        f"--name {quote_bash(name)} "
        f"--cpus {VALHALLA_CPUS} "
        "-p 8002:8002 "
        "-e build_admins=True "
        "-e build_time_zones=True "
        "-e build_tar=True "
        "-e serve_tiles=True "
        f"-v {quote_wsl_path(graph_dir)}:/custom_files "
        f"{quote_bash(VALHALLA_IMAGE)}",
    ])
    run_wsl(command)


def stop_valhalla_container(year: int) -> None:
    run_wsl(f"docker rm -f {quote_bash(container_name(year))} >/dev/null 2>&1 || true", check=False)


def valhalla_test_route() -> dict:
    payload = {"locations": TEST_ROUTE, "costing": "auto", "directions_options": {"units": "kilometers"}}
    response = requests.post(f"{VALHALLA_URL}/route", json=payload, timeout=30)
    response.raise_for_status()
    data = response.json()
    summary = data["trip"]["summary"]
    if summary.get("length", 0) <= 0 or summary.get("time", 0) <= 0:
        raise ValueError(f"Invalid Valhalla route summary: {summary}")
    return summary


def wait_until_valhalla_ready(year: int, max_wait_minutes: int = MAX_WAIT_MINUTES) -> dict:
    deadline = time.time() + max_wait_minutes * 60
    last_error = None
    while time.time() < deadline:
        try:
            return valhalla_test_route()
        except Exception as error:
            last_error = error
            time.sleep(30)
    logs = run_wsl(f"docker logs --tail 80 {quote_bash(container_name(year))}", check=False).stdout
    raise TimeoutError(f"Valhalla did not become ready for {year}. Last error: {last_error}\n\nContainer logs:\n{logs}")


def write_wsl_manifest(year: int, pbf_wsl_path: str, route_summary: dict, started_at: str, finished_at: str) -> str:
    pbf_path = PBF_BY_YEAR[year]
    manifest = {
        "year": year,
        "osm_snapshot_filename": pbf_path.name,
        "osm_snapshot_windows_path": str(pbf_path),
        "osm_snapshot_wsl_path": pbf_wsl_path,
        "osm_snapshot_sha256": sha256_file(pbf_path),
        "graph_path_wsl": wsl_graph_dir(year),
        "valhalla_image": VALHALLA_IMAGE,
        "profile": "auto",
        "study_area": "Austria PBF with Styria analysis focus",
        "graph_build_started_at": started_at,
        "graph_build_finished_at": finished_at,
        "test_route_summary": route_summary,
        "status": "done",
    }
    manifest_json = json.dumps(manifest, indent=2)
    command = f"cat > {quote_wsl_path(wsl_manifest_path(year))} <<'EOF'\n{manifest_json}\nEOF"
    run_wsl(command)
    return wsl_manifest_path(year)

## Build Graphs

Keep `RUN_BUILD = False` for a dry run. Set `RUN_BUILD = True` after the preflight cell works. Start with `BUILD_YEARS = [2025]`; after that validates, switch to `BUILD_YEARS = YEARS`.

In [11]:
for year in BUILD_YEARS:
    started_at = datetime.now().isoformat(timespec="seconds")
    graph_dir = wsl_graph_dir(year)
    manifest_path = wsl_manifest_path(year)

    if graph_manifest_exists(year) and not OVERWRITE:
        print(f"Skip {year}: manifest exists at {manifest_path}")
        write_status_row({
            "year": year,
            "status": "skipped",
            "started_at": started_at,
            "finished_at": datetime.now().isoformat(timespec="seconds"),
            "duration_minutes": 0,
            "graph_path_wsl": graph_dir,
            "manifest_path_wsl": manifest_path,
            "error_message": "existing manifest",
        })
        continue

    print(f"Build {year}: {PBF_BY_YEAR[year].name} -> {graph_dir}")
    if not RUN_BUILD:
        print("Dry run only. Set RUN_BUILD = True to start Docker.")
        continue

    write_status_row({
        "year": year,
        "status": "running",
        "started_at": started_at,
        "finished_at": "",
        "duration_minutes": "",
        "graph_path_wsl": graph_dir,
        "manifest_path_wsl": manifest_path,
        "error_message": "",
    })

    try:
        pbf_wsl_path = copy_pbf_to_wsl(year)
        start_valhalla_container(year)
        route_summary = wait_until_valhalla_ready(year)
        finished_at = datetime.now().isoformat(timespec="seconds")
        manifest_path = write_wsl_manifest(year, pbf_wsl_path, route_summary, started_at, finished_at)
        stop_valhalla_container(year)

        duration_minutes = round((datetime.fromisoformat(finished_at) - datetime.fromisoformat(started_at)).total_seconds() / 60, 2)
        write_status_row({
            "year": year,
            "status": "done",
            "started_at": started_at,
            "finished_at": finished_at,
            "duration_minutes": duration_minutes,
            "graph_path_wsl": graph_dir,
            "manifest_path_wsl": manifest_path,
            "error_message": "",
        })
        print(f"Done {year}: {route_summary}")
    except Exception as error:
        stop_valhalla_container(year)
        finished_at = datetime.now().isoformat(timespec="seconds")
        duration_minutes = round((datetime.fromisoformat(finished_at) - datetime.fromisoformat(started_at)).total_seconds() / 60, 2)
        write_status_row({
            "year": year,
            "status": "failed",
            "started_at": started_at,
            "finished_at": finished_at,
            "duration_minutes": duration_minutes,
            "graph_path_wsl": graph_dir,
            "manifest_path_wsl": manifest_path,
            "error_message": repr(error),
        })
        raise

Build 2015: austria-150101.osm.pbf -> $HOME/gruendungsanalyse/data/routing/valhalla_graphs/2015
Done 2015: {'has_time_restrictions': True, 'has_toll': False, 'has_highway': False, 'has_ferry': False, 'min_lat': 47.058114, 'min_lon': 15.438201, 'max_lat': 47.070316, 'max_lon': 15.464347, 'time': 681.979, 'length': 3.41, 'cost': 1634.706}
Build 2016: austria-160101.osm.pbf -> $HOME/gruendungsanalyse/data/routing/valhalla_graphs/2016
Done 2016: {'has_time_restrictions': True, 'has_toll': False, 'has_highway': False, 'has_ferry': False, 'min_lat': 47.058114, 'min_lon': 15.438201, 'max_lat': 47.070316, 'max_lon': 15.464347, 'time': 681.789, 'length': 3.409, 'cost': 1631.874}
Build 2017: austria-170101.osm.pbf -> $HOME/gruendungsanalyse/data/routing/valhalla_graphs/2017
Done 2017: {'has_time_restrictions': True, 'has_toll': False, 'has_highway': False, 'has_ferry': False, 'min_lat': 47.058114, 'min_lon': 15.438201, 'max_lat': 47.070316, 'max_lon': 15.464347, 'time': 751.579, 'length': 3.405,

In [12]:
read_status()

,year,status,started_at,finished_at,duration_minutes,graph_path_wsl,manifest_path_wsl,error_message
0,2015,done,2026-06-25T21:55:58,2026-06-25T21:57:32,1.57,$HOME/gruendungsanalyse/data/routing/valhalla_...,$HOME/gruendungsanalyse/data/routing/valhalla_...,NaN
1,2016,done,2026-06-25T21:57:33,2026-06-25T21:59:08,1.58,$HOME/gruendungsanalyse/data/routing/valhalla_...,$HOME/gruendungsanalyse/data/routing/valhalla_...,NaN
2,2017,done,2026-06-25T21:59:09,2026-06-25T22:00:45,1.60,$HOME/gruendungsanalyse/data/routing/valhalla_...,$HOME/gruendungsanalyse/data/routing/valhalla_...,NaN
3,2018,done,2026-06-25T22:00:46,2026-06-25T22:02:23,1.62,$HOME/gruendungsanalyse/data/routing/valhalla_...,$HOME/gruendungsanalyse/data/routing/valhalla_...,NaN
4,2019,done,2026-06-25T22:02:24,2026-06-25T22:04:31,2.12,$HOME/gruendungsanalyse/data/routing/valhalla_...,$HOME/gruendungsanalyse/data/routing/valhalla_...,NaN
5,2020,done,2026-06-25T22:04:32,2026-06-25T22:06:40,2.13,$HOME/gruendungsanalyse/data/routing/valhalla_...,$HOME/gruendungsanalyse/data/routing/valhalla_...,NaN
6,2021,done,2026-06-25T22:06:41,2026-06-25T22:08:49,2.13,$HOME/gruendungsanalyse/data/routing/valhalla_...,$HOME/gruendungsanalyse/data/routing/valhalla_...,NaN
7,2022,done,2026-06-25T22:08:50,2026-06-25T22:10:58,2.13,$HOME/gruendungsanalyse/data/routing/valhalla_...,$HOME/gruendungsanalyse/data/routing/valhalla_...,NaN
8,2023,done,2026-06-25T22:10:59,2026-06-25T22:13:08,2.15,$HOME/gruendungsanalyse/data/routing/valhalla_...,$HOME/gruendungsanalyse/data/routing/valhalla_...,NaN
9,2024,done,2026-06-25T22:13:09,2026-06-25T22:15:47,2.63,$HOME/gruendungsanalyse/data/routing/valhalla_...,$HOME/gruendungsanalyse/data/routing/valhalla_...,NaN
